# 🚀 **Getting Started — Fabric Spark Performance Workshop**

Welcome! This notebook is your launchpad for the day. Run through it once before Module 0 to
confirm your environment is ready, seed the bronze Delta tables, and get oriented to the
materials.

**Time required:** ~10–15 minutes (most of it is waiting for the seeding job to finish)

---

## What you'll do here

1. ✅ Verify prerequisites (workspace, attached lakehouse, Fabric Environment)
2. 🧱 Trigger the `seed_lego_delta_tables` Spark Job Definition to build the bronze layer
3. 🔍 Validate the bronze tables landed correctly
4. 🗺️ Get oriented to the LEGO data model and the workshop modules


---

## 🗺️ The LEGO Lakehouse — Data Model

Throughout the workshop we use a synthetic LEGO manufacturing & sales lakehouse generated by
[LakeGen](../LakeGen/). The diagram below shows the full schema — fact tables (events,
orders, transactions) at the center, dimensions (parts, colors, themes, sets) on the
periphery.

![LEGO data model](lego_diagram.png)

> 💡 The biggest fact table is `manufacturing_event` (~8 M rows / ~350 MB). Most performance
> exercises target it because it's the only table large enough to make Spark sweat at lab scale.


---

## 1️⃣ Prerequisites — Quick Check

Before you run anything, confirm:

| Item | How to check |
|------|--------------|
| **Fabric workspace** | You can see this notebook in the workshop Fabric workspace. |
| **Default lakehouse** | A lakehouse named `lego_lakehouse` is **attached** to this notebook (look at the Lakehouse explorer pane on the left). |
| **Spark Environment** | This notebook is attached to a Fabric Spark Environment that has the `LakeGen` and `ArcFlow` wheels installed. |
| **Permissions** | You have write access to the lakehouse (needed to land bronze tables). |

If any of these are missing, ask your instructor before continuing.

Run the next cell — it prints the live values so you can confirm visually.


In [ ]:
# Print runtime context so you can verify the notebook is wired up correctly.
import sys

print("Python version:", sys.version.split()[0])
print("Spark version :", spark.version)
print("App name      :", spark.sparkContext.appName)

# Confirm libraries that the seeding job and modules depend on
for mod in ["lakegen", "arcflow"]:
    try:
        __import__(mod)
        print(f"   ✅ {mod} importable")
    except ImportError as e:
        print(f"   ❌ {mod} NOT available — {e}")
        print(f"      → Add the {mod} wheel to your Fabric Environment and reattach.")


---

## 2️⃣ Seed the Bronze Layer — Run `seed_lego_delta_tables`

The workshop's source data does not live in the lakehouse yet. The first thing you'll do is
run a **Spark Job Definition** that:

1. Generates LakeGen LEGO landing data (mixed JSON + Parquet) into `Files/landing/…`
2. Ingests every table through ArcFlow into bronze Delta tables under the `bronze` schema

### How to trigger it

1. Open the workspace and locate the **Spark Job Definition** named
   **`seed_lego_delta_tables`**.
2. Open it and click **Run** (top right). Select the same Fabric Environment that this
   notebook is attached to.
3. Wait for the job to reach **Succeeded** in the run history. First run takes **~10–15 minutes**
   end-to-end (most of it is generation, not Spark).

> 🔁 **Re-run anytime.** The job is idempotent — re-running drops and rebuilds the bronze
> tables. If you ever want a clean slate, just re-run it.

> ⚠️ **Don't run the `seed_lego_delta_tables_unoptimized` job** unless an instructor tells you
> to. That variant is used as a "before" snapshot for Module 2 — running it overwrites the
> tuned bronze layer with deliberately-bad small files.

While the job runs, you can keep reading. Come back and execute the validation cell once the
job reports **Succeeded**.


### ✅ Validation — confirm the bronze layer is ready

Run the next two cells **after** the seeding job succeeds. They confirm the bronze schema
exists and spot-check a couple of the largest tables.


In [ ]:
# 1. List the tables in the bronze schema — you should see ~15 tables
spark.sql("SHOW TABLES IN bronze").show(50, truncate=False)


In [ ]:
# 2. Spot-check the biggest fact table — used heavily in Modules 1, 2, and 4
me_count = spark.table("bronze.manufacturing_event").count()
print(f"   bronze.manufacturing_event row count: {me_count:,}")

if me_count < 1_000_000:
    print("   ⚠️  Row count is unexpectedly low — re-run the seeding job.")
else:
    print("   ✅ Bronze layer looks healthy. You're ready for Module 0.")


---

## 3️⃣ Workshop Modules — Where to Go Next

The day is structured into one lecture-and-demo module (Module 0) plus four hands-on labs.
Each lab is a single notebook in this workspace.

| Module | Notebook | What you'll learn |
|--------|----------|--------------------|
| **0 — Foundational Theory** | *(instructor-led, no notebook)* | Spark execution architecture, partitions, Catalyst, Delta internals |
| **1 — Performance Diagnostics** | `01-lab-diagnostics.ipynb` | Reading the Spark UI, query plans, and Delta metadata to localize bottlenecks |
| **2 — Configuration & Tuning** | `02-lab-configuration-tuning.ipynb` | OPTIMIZE / compaction, deletion vectors, liquid clustering, data skipping stats |
| **3 — Optimization Patterns** | `03-lab-optimization.ipynb` | Join strategies, caching, AQE, shuffle tuning |
| **4 — Advanced Debugging (Capstone)** | `04-lab-debugging-capstone-v2.ipynb` | Triage 5 reproducible failure modes end-to-end |

### Helper / shared notebooks

| Notebook | What it provides |
|----------|------------------|
| `_benchmark_utils.ipynb` | `benchmark_op(scenario, state, spark)` context manager and `get_table_metrics()` / `show_metrics()` helpers used across labs. Don't run it standalone — the lab notebooks `%run` it. |

### Reference materials

- **`workshop-plan.md`** — full agenda, timing, and learning objectives.
- **`lab-technical-plan.md`** — technical specs per lab.
- **`04-lab-debugging-capstone-v2-speaker-notes.md`** — instructor speaker notes for the capstone.


---

## 4️⃣ Spark UI — Know Where to Look

You'll be sent to the Spark UI repeatedly throughout the day. Bookmark these tabs now:

| Tab | What it tells you |
|-----|--------------------|
| **Jobs** | High-level: which jobs ran, how long, status. Start here. |
| **Stages** | Drill into a job. Look at task duration min/median/max for skew. |
| **SQL / DataFrame** | Per-query physical plan with row counts and timing per node. |
| **Storage** | Cached RDD/DataFrame footprint — useful for Module 3 caching exercises. |
| **Executors** | Per-executor memory / GC / shuffle. First stop for OOM debugging. |

To open the Spark UI for any running query in Fabric: click the **Spark monitoring** link that
appears below the cell after an action runs.


---

## 5️⃣ Troubleshooting

| Symptom | Likely cause / fix |
|---------|--------------------|
| `ModuleNotFoundError: lakegen` or `arcflow` | The attached Fabric Environment is missing the wheel. Ask the instructor to attach the workshop environment, then **Detach + Reattach** this notebook. |
| `Table or view 'bronze.manufacturing_event' cannot be found` | The seeding job hasn't run (or it failed). Re-run `seed_lego_delta_tables` and watch the run history for errors. |
| Cell hangs on the first Spark call for minutes | Cold session start in Fabric — first cell after a long idle takes 1–3 minutes to warm up. Subsequent cells are fast. |
| Bronze tables exist but row counts look tiny | The seeding job ran with the wrong scale or was interrupted. Re-run it. |
| `%run _benchmark_utils` fails | The helper notebook isn't in the same workspace. Confirm `_benchmark_utils.ipynb` is uploaded to the workspace. |

---

## ✅ You're ready

When the validation cells above pass and you've taken a minute to look at the LEGO data model
diagram, you're set for Module 0. See you in the room.
